In [7]:
#!/usr/bin/env python
# coding: utf-8

import os
import pandas as pd
import numpy as np
from glob import glob

# ====================================
# CONFIGURATION
# ================
OUTPUT_PATH = "cleaned_data/crss_vehicle_cleaned.csv"

VARS_OF_INTEREST = [
    "CASENUM", "VEH_NO",
    "VSURCOND", "VTRAFCON", "VPAVETYP", "ROUTE", "BODY_TYP", "TOW_VEH",
    "TRLR1GVWR", "TRLR2GVWR", "TRLR3GVWR", "V_CONFIG", "CARGO_BT",
    "TRAV_SP", "M_HARM", "SPEEDREL", "VTRAFWAY", "VNUM_LAN",
    "VSPD_LIM", "VALIGN", "VPROFILE", "VTCONT_F", "P_CRASH1", "P_CRASH2"
]

# ====================================
# LOAD AND CLEAN EACH YEAR
# ====================================
def load_crss_vehicle_file(year_folder):
    """
    Load the CRSS vehicle file for a given year folder.
    Handles capitalization differences (VEHICLE.csv, Vehicle.csv, vehicle.csv).
    """
    candidates = glob(os.path.join(year_folder, "[Vv][Ee][Hh][Ii][Cc][Ll][Ee].csv"))
    if not candidates:
        print(f" No vehicle file found in {year_folder}")
        return None
    file_path = candidates[0]

    try:
        df = pd.read_csv(file_path, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="latin1", low_memory=False)

    df.columns = df.columns.str.upper().str.strip()
    df["YEAR"] = int(os.path.basename(year_folder))
    return df

# ====================================
# VARIABLE HARMONIZATION + CLEANING
# ====================================
def harmonize_crss_values(df):
    """Apply variable harmonization and missing normalization for CRSS."""
    df = df.copy()

    # --- TRAV_SP (Travel Speed) ---
    def clean_trav_sp(row):
        val, year = row.get("TRAV_SP"), row.get("YEAR")
        if pd.isna(val): return np.nan
        try: val = int(val)
        except: return np.nan
        if val == 0: return 0
        elif 1 <= val <= 151: return val
        elif val == 997: return 152
        elif val in [998, 999]: return np.nan
        return np.nan
    if "TRAV_SP" in df.columns:
        df["TRAV_SP"] = df.apply(clean_trav_sp, axis=1)

    if "SPEEDREL" in df.columns:
        df["SPEEDREL"] = pd.to_numeric(df["SPEEDREL"], errors="coerce")
        df.loc[df["SPEEDREL"].isin([8,9]), "SPEEDREL"] = np.nan

    for col, missing_codes in [("VSURCOND",[98,99]), ("VTRAFCON",[97,98,99]), ("VPAVETYP",[8,9])]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col].isin(missing_codes), col] = np.nan

    if "M_HARM" in df.columns:
        df["M_HARM"] = pd.to_numeric(df["M_HARM"], errors="coerce")
        df.loc[df["M_HARM"].isin([98,99]), "M_HARM"] = np.nan

    if "VTRAFWAY" in df.columns:
        df["VTRAFWAY"] = pd.to_numeric(df["VTRAFWAY"], errors="coerce")
        df.loc[df["VTRAFWAY"].isin([8,9]), "VTRAFWAY"] = np.nan

    if "VNUM_LAN" in df.columns:
        df["VNUM_LAN"] = pd.to_numeric(df["VNUM_LAN"], errors="coerce")
        df.loc[df["VNUM_LAN"].isin([8,9]), "VNUM_LAN"] = np.nan

    if "VSPD_LIM" in df.columns:
        df["VSPD_LIM"] = pd.to_numeric(df["VSPD_LIM"], errors="coerce")
        df.loc[df["VSPD_LIM"].isin([98,99]), "VSPD_LIM"] = np.nan

    if "VALIGN" in df.columns:
        df["VALIGN"] = pd.to_numeric(df["VALIGN"], errors="coerce")
        df.loc[df["VALIGN"].isin([8,9]), "VALIGN"] = np.nan

    if "VPROFILE" in df.columns:
        df["VPROFILE"] = pd.to_numeric(df["VPROFILE"], errors="coerce")
        df.loc[df["VPROFILE"].isin([8,9]), "VPROFILE"] = np.nan

    if "VTCONT_F" in df.columns:
        df["VTCONT_F"] = pd.to_numeric(df["VTCONT_F"], errors="coerce")
        df.loc[df["VTCONT_F"].isin([8,9]), "VTCONT_F"] = np.nan

    for col in ["P_CRASH1", "P_CRASH2"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col].isin([98,99]), col] = np.nan

    if "BODY_TYP" in df.columns:
        df["BODY_TYP"] = pd.to_numeric(df["BODY_TYP"], errors="coerce")
        df.loc[df["BODY_TYP"].isin([97,98,99]), "BODY_TYP"] = np.nan

    if "ROUTE" in df.columns:
        df["ROUTE"] = pd.to_numeric(df["ROUTE"], errors="coerce")
        df.loc[df["ROUTE"].isin([97,98,99]), "ROUTE"] = np.nan

    if "TOW_VEH" in df.columns:
        df["TOW_VEH"] = pd.to_numeric(df["TOW_VEH"], errors="coerce")
        df.loc[df["TOW_VEH"].isin([7,8,9]), "TOW_VEH"] = np.nan

    for col in ["TRLR1GVWR", "TRLR2GVWR", "TRLR3GVWR"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df.loc[df[col].isin([98,99]), col] = np.nan

    if "V_CONFIG" in df.columns:
        df["V_CONFIG"] = pd.to_numeric(df["V_CONFIG"], errors="coerce")
        df.loc[df["V_CONFIG"].isin([88,99]), "V_CONFIG"] = np.nan

    if "CARGO_BT" in df.columns:
        df["CARGO_BT"] = pd.to_numeric(df["CARGO_BT"], errors="coerce")
        df.loc[df["CARGO_BT"].isin([98,99]), "CARGO_BT"] = np.nan

    return df

# ====================================
# COMBINE ALL YEARS
# ====================================
def combine_crss_data(base_dir):
    year_folders = sorted([f.path for f in os.scandir(base_dir) if f.is_dir()])
    all_dfs = []
    for folder in year_folders:
        df = load_crss_vehicle_file(folder)
        if df is None:
            continue

        # select available columns
        cols = [c for c in VARS_OF_INTEREST if c in df.columns]
        df = df[cols + ["YEAR"]]

        df = harmonize_crss_values(df)

        # Create unique ID: YEAR + CASENUM
        if "CASENUM" in df.columns:
            df["ID"] = df["YEAR"].astype(str) + "_" + df["CASENUM"].astype(str)
            df.insert(0, "ID", df.pop("ID"))

        all_dfs.append(df)
        print(f" Processed {os.path.basename(folder)}: {df.shape[0]} rows, {df.shape[1]} cols")

    combined = pd.concat(all_dfs, ignore_index=True)
    return combined

# ====================================
# DATA QUALITY CHECK
# ====================================
def summarize_data_quality(df):
    print("\n=== DATA QUALITY SUMMARY ===")
    missing_pct = df.isna().mean() * 100
    print("Missingness (%):")
    print(missing_pct.sort_values(ascending=False))
    print("\nRanges / Unique Values:")
    for col in df.columns:
        if df[col].dtype in [np.int64, np.float64]:
            print(f"{col}: min={df[col].min()}, max={df[col].max()}")
        else:
            print(f"{col}: {df[col].nunique()} unique values")

# ====================================
# DUPLICATE CHECK
# ====================================
def check_duplicate_keys(df):
    key_cols = ["CASENUM", "VEH_NO"]
    if "PER_NO" in df.columns:
        key_cols.append("PER_NO")
    duplicates = df[df.duplicated(subset=key_cols, keep=False)]
    print(f"\nDuplicate key combinations ({', '.join(key_cols)}): {len(duplicates)}")
    return duplicates

# ====================================
# MAIN SCRIPT
# ====================================
if __name__ == "__main__":
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    crss_clean = combine_crss_data(BASE_DIR)
    summarize_data_quality(crss_clean)

    check_duplicate_keys(crss_clean)

    crss_clean.to_csv(OUTPUT_PATH, index=False)
    print(f"\n Cleaned CRSS data saved to: {OUTPUT_PATH}")


 Processed 2016: 82149 rows, 21 cols
 Processed 2017: 97625 rows, 21 cols
 Processed 2018: 86105 rows, 21 cols
 Processed 2019: 96717 rows, 21 cols
 Processed 2020: 94718 rows, 24 cols
 Processed 2021: 95785 rows, 24 cols
 Processed 2022: 94756 rows, 24 cols
 Processed 2023: 87461 rows, 24 cols

=== DATA QUALITY SUMMARY ===
Missingness (%):
TRAV_SP      52.698840
TRLR1GVWR    50.780073
TRLR2GVWR    49.349939
TRLR3GVWR    49.313356
VNUM_LAN     26.964734
VTRAFWAY     16.712271
VSPD_LIM     14.146435
VPROFILE     14.130931
VTRAFCON     10.391587
VTCONT_F     10.099331
VALIGN        5.439838
VSURCOND      4.997579
CARGO_BT      2.981847
SPEEDREL      2.479478
V_CONFIG      2.011380
P_CRASH2      1.911151
BODY_TYP      1.768627
P_CRASH1      1.758972
TOW_VEH       0.182643
M_HARM        0.045559
CASENUM       0.000000
YEAR          0.000000
VEH_NO        0.000000
ID            0.000000
dtype: float64

Ranges / Unique Values:
ID: 417335 unique values
CASENUM: min=201600014311, max=202305779